In [1]:
import numpy as np
import scipy.sparse as sp
import time
from dolphindes.cvxopt import DenseSharedProjQCQP, OptimizationHyperparameters

# conda clean --all

In [2]:
# Physical constants
epsilon_0 = 8.854e-12 # F/m
mu_0 = 4e-7 * np.pi # H/m
c = 1/np.sqrt(epsilon_0 * mu_0) # speed of light in vacuum

E0 = 1e3 # plane wave amplitude in V/m

frequency = 2.45e9 # frequency in Hz
wavelength = c / frequency # wavelength in m
print(f"Wavelength: {wavelength} m")

ka = 1 # electrical size of the antenna
k = 2 * np.pi / wavelength # wavenumber
a = ka / k # antenna circumradius
print(f"Antenna circumradius: {a} m")

omega = 2 * np.pi * frequency # angular frequency

conductivity_reduction_factor = 1 # factor to reduce conductivity for testing purposes
copper_conductivity = conductivity_reduction_factor*5.96e7 # S/m
copper_permittivity = 1 + 1j * copper_conductivity / (omega * epsilon_0) # copper permittivity in dimensionless units
print(f"Copper permittivity: {copper_permittivity:.2e}")

Wavelength: 0.1223655664053944 m
Antenna circumradius: 0.019475084757658086 m
Copper permittivity: 1.00e+00+4.37e+08j


In [3]:
# Calculate surface impedance
delta = np.sqrt(2 / (omega * mu_0 * copper_conductivity)) # skin depth in m
Zs = (1 + 1j) / (copper_conductivity * delta)
# Zs = 1 / (copper_conductivity * delta)
print(f"Surface impedance: {Zs} Ω")

# Load matrices from text files
Lmat = np.loadtxt(r"Lmat_matrix.txt", delimiter=',') # lossy matrix
R0 = np.loadtxt(r"R0_matrix.txt", delimiter=',') # radiated power matrix
X0 = np.loadtxt(r"X0_matrix.txt", delimiter=',') # reactive power matrix
Vinc = E0*np.loadtxt(r"V_vector.txt", delimiter=',') # voltage excitation vector
Zmat = Zs * Lmat # material impedance matrix
Z0 = R0 + 1j*X0 # free-space impedance matrix
Ztot = Z0 + Zmat # total impedance matrix
Rmat = np.real(Zmat) # rezistivity matrix

Ndes = Lmat.shape[0] # number of design variables
print("Number of design variables: ", Ndes)

Vzero = np.zeros((Ndes,), dtype=complex) # zero vector

Obj = -0.5 * R0
obj = Vzero/2
obj0 = 0

# full plate performance
Ifull = np.linalg.solve(Ztot, Vinc) # current distribution for full plate
Pfull = -np.real(Ifull.conj().T @ Obj @ Ifull) + 2*np.real(Ifull.conj().T @ obj) + obj0 # objective for full plate
print(f"Objective for full plate: {Pfull:.6e} W")

Surface impedance: (0.012739130327240646+0.012739130327240646j) Ω
Number of design variables:  195
Objective for full plate: 1.010821e+01 W


In [4]:
# Assuming Ndes is OP.BF.nUnknowns and is an odd integer
k = (Ndes - 1) // 2

# 1. Upper block: Identity matrix of size (k + 1)
upper = np.eye(k + 1)

# 2. Lower block: Flipped identity of size k, followed by a column of zeros
# np.fliplr(np.eye(k)) creates the anti-diagonal matrix
lower_left = np.fliplr(np.eye(k))
lower_right = np.zeros((k, 1))
lower = np.hstack([lower_left, lower_right])

# 3. Vertically stack them to create C
C = np.vstack([upper, lower])
C = np.eye(Ndes) # skip transformation for testing purposes

# lossy matrix Cholesky factorization
Lchol = np.linalg.cholesky(C.conj().T @Lmat @ C).conj().T

# Mfactor = C @ np.linalg.inv(Lchol)
Mfactor = np.eye(Ndes) # skip transformation for testing purposes
# Mfactor = C

Vincf = Mfactor.conj().T @ Vinc
Ztotf = Mfactor.conj().T @ Ztot @ Mfactor
Z0f = Mfactor.conj().T @ Z0 @ Mfactor
Zmatf = Mfactor.conj().T @ Zmat @ Mfactor
Objf = Mfactor.conj().T @ Obj @ Mfactor
objf = Mfactor.conj().T @ obj
obj0f = obj0

Nfac = Z0f.shape[0]
iVec = np.zeros(Nfac, dtype=complex) # particular solution to satisfy the fixed current constraint (zero for now, can be used to satisfy a fixed current constraint by setting the iBFfixed row to np.linalg.solve(Z11, Vinc1) and the transformation matrix to have -np.linalg.solve(Z11, Z12) in the iBFfixed row)

In [5]:
# translate QCQP matrices to Dolphindes notation
Umat = 1j * Ztotf.conj() # Dolphindes U matrix
eVec = -1j*Vincf.conj() / 2 # Dolphindes e vector
Bmat = Objf # quadratic objective matrix
bVec = objf # linear objective vector
beta = obj0f # constant objective term

In [6]:
# preconditioner for projection constraints
G0 = Z0f # Green's function matrix
D = np.diag(np.diag(Z0f)) # Green's matrix diagonal
X = np.linalg.inv(Zmatf) # material admittance matrix
H0 = G0 - D
Y = np.linalg.solve(np.eye(Nfac) + X @ D, X)
iVec = X @ Vincf

S = np.eye(Nfac) + Y @ H0 # preconditioner matrix for projection constraints
q = -Y @ G0 @ iVec

P0 = np.linalg.inv(S) @ Ztotf

# print projector error
pErr = np.linalg.norm(S @ P0 - Ztotf, ord='fro')/np.linalg.norm(Ztotf, ord='fro')
print(f"Projector error: {pErr:.2e}")

# # plot maginitude and phase of projector matrix P0
# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 5))
# plt.subplot(1, 2, 1)
# plt.imshow(np.log(np.abs(P0)), cmap='viridis')
# plt.colorbar()
# plt.title('Magnitude of Projector P0')
# plt.subplot(1, 2, 2)
# plt.imshow(np.angle(P0), cmap='twilight')
# plt.colorbar()
# plt.title('Phase of Projector P0')
# plt.show()

# # plot P0 @ S eigenvalues
# plt.figure(figsize=(6, 6))
# eigenvalues = np.linalg.eigvals((1J * S.conj()) @ (-1J *P0.conj()))
# plt.scatter(eigenvalues.real, eigenvalues.imag, color='blue', marker='o')
# plt.axhline(0, color='gray', linestyle='--')
# plt.axvline(0, color='gray', linestyle='--')
# plt.xlabel('Real Part')
# plt.ylabel('Imaginary Part')
# plt.title('Eigenvalues of P0 @ S')
# plt.grid()
# plt.show() 

sVec = np.linalg.solve(S, q)
tVec = sVec + iVec
Vtest = Ztotf @ tVec
print(f"voltage error: {np.max(np.abs(Vtest - Vincf))/np.max(np.abs(Vincf)):.2e}")

print(f"asymmetric error of preconditioner: {np.linalg.norm(S-S.T, ord='fro')/np.linalg.norm(S, ord='fro'):.2e}")

Objp = Objf
objp = objf - Objf @ iVec
obj0p = obj0f - np.real(iVec.conj().T  @ Objf @ iVec) + 2*np.real(iVec.conj().T @ objf)

# Ptr = -np.real(tVec.conj().T @ Objf @ tVec)
# Ptr  = -np.real((sVec + iVec).conj().T @ Objf @ (sVec + iVec)) + 2*np.real((sVec + iVec).conj().T @ objf) + obj0f
Ptr = -np.real(sVec.conj().T @ Objp @ sVec) + 2*np.real(sVec.conj().T @ objp) + obj0p
print(f"power error with preconditioner: {np.abs(Ptr - Pfull)/np.abs(Pfull):.2e}")

Umat = -S
eVec = -q/2
Bmat = Objp
bVec = objp
beta = obj0p

Projector error: 1.05e-13
voltage error: 5.38e-10
asymmetric error of preconditioner: 2.14e-09
power error with preconditioner: 6.29e-11


In [ ]:
# make the Plist to contain matrices P0 and 1j*P0
Plist = [P0, 1j*P0]

QCQP = DenseSharedProjQCQP(Bmat, bVec, beta,
                            Umat, eVec,
                             Plist, verbose = 1
                                )
# print fist elements of all inputs
# print(f"Bmat: {Bmat[0,0]}, bVec: {bVec[0]}, beta: {beta}, Umat: {Umat[0,0]}, eVec: {eVec[0] }")

t1 = time.time()
lags_init = np.zeros((2,))
# lags_init[0] = -1
lags_init[0] = -0.61832415
lags_init[1] = -1.13213161
# New interface for specifying optimization hyperparameters. https://dolphindes.readthedocs.io/en/latest/api/dolphindes.cvxopt.OptimizationHyperparameters.html
opt_params = OptimizationHyperparameters(
    opttol=1e-12,                  
    gradConverge=True,           
    min_inner_iter=10,             
    max_restart=30,                
    penalty_ratio=1e-2,           
    penalty_reduction=0.1,        
    break_iter_period=5,          
    verbose=1)

# Note: 'newton' is usually faster than BFGS, both are prety fast with so few constraints.
result = QCQP.solve_current_dual_problem(method = 'bfgs', init_lags = lags_init, opt_params = opt_params)
print(f"bound: {result[0]:.4e}, time: {time.time()-t1}s, Lagrange multipliers: {QCQP.current_lags}")

Precomputed 2 A matrices and Fs vectors.
Optimizer initialized with parameters:
opttol: 1e-12
gradConverge: True
min_inner_iter: 10
max_restart: 30
penalty_ratio: 0.01
penalty_reduction: 0.1
break_iter_period: 5
verbose: 1
Starting optimization with x0 = [-0.5589 -0.9326]
Outer iteration 0, penalty_ratio = 0.01, opt_fx = 3316455.415169424
iter_num: 5, prev_fx: inf, opt_fx: 3146091.4628033824, opttol: 1e-12
iter_num: 10, prev_fx: 3146091.4628033824, opt_fx: 3146086.539885133, opttol: 1e-12
iter_num: 15, prev_fx: 3146086.539885133, opt_fx: 3146086.539883571, opttol: 1e-12
Outer iteration 1, penalty_ratio = 0.001, opt_fx = 3146086.539883571
iter_num: 5, prev_fx: 3146086.539885133, opt_fx: 3146086.539886082, opttol: 1e-12
bound: 3.1461e+06, time: 0.2842123508453369s, Lagrange multipliers: [-0.61832415 -1.13213161]
